In [1]:
import pandas as pd
import numpy as np

autorzy = pd.DataFrame({
    'autor_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'imie':     ['Olga', 'Stanislaw', 'Andrzej', 'Wislawa', 'Ryszard',
                 'Dorota', 'Szczepan', 'Jacek'],
    'nazwisko': ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuscinski',
                 'Maslowska', 'Twardoch', 'Dehnel'],
    'kraj':     ['Polska'] * 8,
    'nagrody':  ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter',
                 'Polityka', 'NIKE', 'NIKE']
})

ksiazki = pd.DataFrame({
    'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    'autor_id':   [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
    'tytul': ['Ksiegi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada',
              'Wiedzmin', 'Narrenturm', 'Wiersze wybrane', 'Podroze z Herodotem',
              'Wojna polsko-ruska', 'Morfina', 'Krol', 'Lala'],
    'kategoria': ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi',
                  'fantasy', 'fantasy', 'poezja', 'reportaz',
                  'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
    'cena':   [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00,
               52.00, 38.00, 48.00, 59.00, 44.00],
    'strony': [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})

np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
    'zam_id':     range(5001, 5001 + n),
    'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
    'ilosc':      np.random.randint(1, 6, size=n),
    'data':       pd.to_datetime('2026-01-01') +
                  pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
    'kanal':      np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=[0.6, 0.3, 0.1]),
    'miasto':     np.random.choice(['Warszawa', 'Krakow', 'Wroclaw', 'Gdansk', 'Poznan'], size=n)
})

print(f'Autorzy:    {autorzy.shape}')
print(f'Ksiazki:    {ksiazki.shape}')
print(f'Zamowienia: {zamowienia.shape}')

Autorzy:    (8, 5)
Ksiazki:    (12, 6)
Zamowienia: (80, 6)


In [2]:
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')
print(f'Shape: {ksiazki_z_autorami.shape}')
ksiazki_z_autorami.head()

Shape: (12, 10)


,ksiazka_id,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,101,1,Ksiegi Jakubowe,historyczna,79.9,912,Olga,Tokarczuk,Polska,Nobel
1,102,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
2,103,2,Solaris,sci-fi,39.0,198,Stanislaw,Lem,Polska,SFF
3,104,2,Cyberiada,sci-fi,42.0,295,Stanislaw,Lem,Polska,SFF
4,105,3,Wiedzmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


In [3]:
ksiazki_test = pd.concat([
    ksiazki,
    pd.DataFrame({'ksiazka_id': [999], 'autor_id': [999],
                  'tytul': ['Ksiazka bez autora'], 'kategoria': ['test'],
                  'cena': [30.0], 'strony': [100]})
], ignore_index=True)

inner = ksiazki_test.merge(autorzy, on='autor_id', how='inner')
left  = ksiazki_test.merge(autorzy, on='autor_id', how='left')
right = ksiazki_test.merge(autorzy, on='autor_id', how='right')
outer = ksiazki_test.merge(autorzy, on='autor_id', how='outer')

print(f'inner: {inner.shape[0]} wierszy')
print(f'left:  {left.shape[0]} wierszy')
print(f'right: {right.shape[0]} wierszy')
print(f'outer: {outer.shape[0]} wierszy')

inner: 12 wierszy
left:  13 wierszy
right: 12 wierszy
outer: 13 wierszy


In [4]:
audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True)
audyt['_merge'].value_counts()

_merge
both          12
left_only      1
right_only     0
Name: count, dtype: int64

In [5]:
pelne = (
    zamowienia
    .merge(ksiazki, on='ksiazka_id')
    .merge(autorzy, on='autor_id')
)
print(f'Pelna tabela: {pelne.shape}')
pelne.head()

Pelna tabela: (80, 15)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,5001,102,5,2026-04-05,web,Poznan,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
1,5002,107,2,2026-02-08,web,Poznan,4,Wiersze wybrane,poezja,35.0,180,Wislawa,Szymborska,Polska,Nobel
2,5003,111,1,2026-04-19,web,Poznan,7,Krol,historyczna,59.0,688,Szczepan,Twardoch,Polska,NIKE
3,5004,109,4,2026-02-24,web,Warszawa,6,Wojna polsko-ruska,obyczajowa,38.0,153,Dorota,Maslowska,Polska,Polityka
4,5005,105,2,2026-01-10,web,Warszawa,3,Wiedzmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


In [6]:
pelne['wartosc']          = pelne['ilosc'] * pelne['cena']
pelne['miesiac']          = pelne['data'].dt.month
pelne['autor_pelne']      = pelne['imie'] + ' ' + pelne['nazwisko']
pelne['kategoria_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania')
pelne[['tytul', 'cena', 'ilosc', 'wartosc', 'miesiac', 'autor_pelne', 'kategoria_cenowa']].head()

,tytul,cena,ilosc,wartosc,miesiac,autor_pelne,kategoria_cenowa
0,Bieguni,49.0,5,245.0,4,Olga Tokarczuk,tania
1,Wiersze wybrane,35.0,2,70.0,2,Wislawa Szymborska,tania
2,Krol,59.0,1,59.0,4,Szczepan Twardoch,droga
3,Wojna polsko-ruska,38.0,4,152.0,2,Dorota Maslowska,tania
4,Wiedzmin,45.0,2,90.0,1,Andrzej Sapkowski,tania


In [7]:
print(f'Liczba zamowien:            {len(pelne)}')
print(f'Laczny przychod:            {pelne["wartosc"].sum():.2f} zl')
print(f'Srednia wartosc zamowienia: {pelne["wartosc"].mean():.2f} zl')
print(f'Unikalne tytuly:            {pelne["tytul"].nunique()}')

Liczba zamowien:            80
Laczny przychod:            11963.80 zl
Srednia wartosc zamowienia: 149.55 zl
Unikalne tytuly:            12


In [8]:
pelne.groupby('kategoria')['wartosc'].sum().sort_values(ascending=False)

kategoria
historyczna    4136.8
sci-fi         2334.0
obyczajowa     1999.0
fantasy        1930.0
reportaz       1144.0
poezja          420.0
Name: wartosc, dtype: float64

In [9]:
pelne.groupby('kategoria')['wartosc'].agg(['count', 'mean', 'sum']).round(2)

,count,mean,sum
kategoria,,,
fantasy,14,137.86,1930.0
historyczna,22,188.04,4136.8
obyczajowa,17,117.59,1999.0
poezja,7,60.00,420.0
reportaz,5,228.80,1144.0
sci-fi,15,155.60,2334.0


In [10]:
pelne.groupby('autor_pelne').agg(
    liczba_zamowien=('zam_id',  'count'),
    laczna_sprzedaz=('wartosc', 'sum'),
    sredni_ilosc   =('ilosc',   'mean')
).round(2).sort_values('laczna_sprzedaz', ascending=False)

,liczba_zamowien,laczna_sprzedaz,sredni_ilosc
autor_pelne,,,
Olga Tokarczuk,18,3389.8,2.72
Stanislaw Lem,15,2334.0,3.80
Andrzej Sapkowski,14,1930.0,2.86
Szczepan Twardoch,11,1580.0,2.91
Ryszard Kapuscinski,5,1144.0,4.40
Jacek Dehnel,5,748.0,3.40
Wislawa Szymborska,7,420.0,1.71
Dorota Maslowska,5,418.0,2.20


In [11]:
pelne.groupby(['kategoria', 'kanal'])['wartosc'].sum()

kategoria    kanal    
fantasy      aplikacja     475.0
             telefon       225.0
             web          1230.0
historyczna  aplikacja    1988.5
             telefon       527.4
             web          1620.9
obyczajowa   aplikacja     586.0
             telefon       136.0
             web          1277.0
poezja       aplikacja      70.0
             telefon        70.0
             web           280.0
reportaz     aplikacja     416.0
             telefon       260.0
             web           468.0
sci-fi       aplikacja     534.0
             telefon       279.0
             web          1521.0
Name: wartosc, dtype: float64

In [12]:
pelne['wartosc_kat']            = pelne.groupby('kategoria')['wartosc'].transform('sum')
pelne['udzial_w_kategorii_pct'] = (pelne['wartosc'] / pelne['wartosc_kat'] * 100).round(2)
pelne.groupby('kategoria')['udzial_w_kategorii_pct'].sum().round(1)

kategoria
fantasy        100.0
historyczna    100.0
obyczajowa     100.0
poezja         100.0
reportaz       100.0
sci-fi         100.0
Name: udzial_w_kategorii_pct, dtype: float64

In [13]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac',
               values='wartosc', aggfunc='sum', fill_value=0)

miesiac,1,2,3,4
kategoria,,,,
fantasy,595.0,410.0,270.0,655.0
historyczna,1342.5,1790.9,634.4,369.0
obyczajowa,301.0,623.0,274.0,801.0
poezja,105.0,70.0,140.0,105.0
reportaz,416.0,0.0,728.0,0.0
sci-fi,765.0,615.0,672.0,282.0


In [14]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac',
               values='wartosc', aggfunc='sum', fill_value=0,
               margins=True, margins_name='RAZEM')

miesiac,1,2,3,4,RAZEM
kategoria,,,,,
fantasy,595.0,410.0,270.0,655.0,1930.0
historyczna,1342.5,1790.9,634.4,369.0,4136.8
obyczajowa,301.0,623.0,274.0,801.0,1999.0
poezja,105.0,70.0,140.0,105.0,420.0
reportaz,416.0,0.0,728.0,0.0,1144.0
sci-fi,765.0,615.0,672.0,282.0,2334.0
RAZEM,3524.5,3508.9,2718.4,2212.0,11963.8


In [15]:
pd.crosstab(pelne['kanal'], pelne['miasto'])

miasto,Gdansk,Krakow,Poznan,Warszawa,Wroclaw
kanal,,,,,
aplikacja,7,1,5,6,4
telefon,1,4,1,3,2
web,5,9,14,9,9


In [16]:
(pd.crosstab(pelne['kanal'], pelne['miasto'], normalize='index') * 100).round(1)

miasto,Gdansk,Krakow,Poznan,Warszawa,Wroclaw
kanal,,,,,
aplikacja,30.4,4.3,21.7,26.1,17.4
telefon,9.1,36.4,9.1,27.3,18.2
web,10.9,19.6,30.4,19.6,19.6


In [17]:
ranking = (
    pelne.groupby(['kategoria', 'autor_pelne'])['wartosc']
    .sum()
    .reset_index()
)
ranking.groupby('kategoria').apply(lambda g: g.nlargest(1, 'wartosc')).reset_index(drop=True)

,autor_pelne,wartosc
0,Andrzej Sapkowski,1930.0
1,Olga Tokarczuk,2556.8
2,Olga Tokarczuk,833.0
3,Wislawa Szymborska,420.0
4,Ryszard Kapuscinski,1144.0
5,Stanislaw Lem,2334.0


In [18]:
pelne.groupby('ksiazka_id').agg(
    strony  =('strony', 'first'),
    sr_ilosc=('ilosc',  'mean')
).corr()

,strony,sr_ilosc
strony,1.000000,-0.135801
sr_ilosc,-0.135801,1.000000


In [19]:
pelne.groupby('kanal')['wartosc'].mean().sort_values(ascending=False).round(2)

kanal
aplikacja    176.93
web          139.06
telefon      136.13
Name: wartosc, dtype: float64